# DFAnalyzer Simple Example

This notebook will guide you to load a trace file generated by DFTracer and analyze the trace events using Dask.

In [ ]:
%load_ext autoreload
%autoreload 2

: 

## System imports for the notebook

In [2]:

import os
from pathlib import Path
import sys

## We add the analysis code to path so that we can run this in dev mode.

In [3]:
app_root = str(Path(os.getcwd()).parent.parent)
sys.path.insert(0, app_root)

In [5]:
sys.path

['/usr/WS2/sinurat1/dftracer',
 '/collab/usr/gapps/python/build/spack-toss4.2/var/spack/environments/python310/._view/i74j3perqsso644fvgbrjnzhyuvpn42l/lib/python310.zip',
 '/collab/usr/gapps/python/build/spack-toss4.2/var/spack/environments/python310/._view/i74j3perqsso644fvgbrjnzhyuvpn42l/lib/python3.10',
 '/collab/usr/gapps/python/build/spack-toss4.2/var/spack/environments/python310/._view/i74j3perqsso644fvgbrjnzhyuvpn42l/lib/python3.10/lib-dynload',
 '',
 '/usr/workspace/sinurat1/ml_workloads/.venv/analyzer/lib/python3.10/site-packages',
 '__editable__.dlio_benchmark-2.0.0.finder.__path_hook__',
 '/usr/workspace/sinurat1/ml_workloads/.venv/analyzer/lib/python3.10/site-packages/setuptools/_vendor']

## Imports for the notebook

This may take some time as it initializes Dask.

In [4]:
# Importing DFAnalyzer
import dfanalyzer
print(dfanalyzer.__file__)
from dfanalyzer.main import DFAnalyzer, update_dft_configuration, setup_logging, setup_dask_cluster, reset_dask_cluster

/usr/WS2/sinurat1/dfanalyzer/dfanalyzer/__init__.py


ModuleNotFoundError: No module named 'dfanalyzer.main'

## Initialize DFAnalyzer Configuration

In this function, we can tune DFAnalyzer for the analysis. For example, we can tune number of workers, connect to existing dask cluster, etc.

In [ ]:
conf = update_dft_configuration(verbose=True, workers=4, log_file=f"./df_{os.getenv('USER')}.log")

## This methods sets up logging for DFAnalyzer.

This is needed for debugging and progress tracking. All prints seen in the following cells are configured in this method.

In [ ]:
setup_logging()

## Setup dask cluster.

In this example, we use Dask Local cluster which will use multiprocessing on the same node where the notebook is running to run its workers.

**NOTE:** If your running on Remote VSCode on a cluster, you can tunnel the port and open it locally.

In [ ]:
setup_dask_cluster()

On clicking the link, you will see a daskboard like this. [Dask Daskboard Image](images/dask-dashboard-load.png)

## Reset Dask Cluster

In case you have an error and want to clean the cluster for fresh analysis. You can run this.

In [ ]:
reset_dask_cluster()

## Load the DFAnalyzer Trace

The DFAnalyzer class take a regex string as input. For example, "{app_root}/examples/dfanalyzer/*.pfw.gz"

In [ ]:
analyzer = DFAnalyzer(f"/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/montage/pegasus-2mass-2deg/node-1/v1/COMPACT/*.pfw.gz")

## Analyze the events

1. The dask dataframe is stored at `analyzer.events`. 
2. We can run dask queries on this dataframe.

In [ ]:
analyzer.events.head()

In [ ]:
analyzer.file_hash.head()

In [ ]:
analyzer.host_hash.head()

### Summary 

DFAnalyzer supports a summary utility that gives a brief summary of the job and its I/O access behavior.

In [ ]:
analyzer.metadata.head()

In [ ]:
items = analyzer.summary()
items

### Timeline plots

We support two timeline plots:
1. how I/O time and I/O bandwidth changes over time.
2. how transfer size changes over time.

In [ ]:
fig, ax1, ax2 = analyzer.plots.time_bw_timeline(
    bw_unit='gb',
    figsize=(8, 3),
    line1_label='POSIX I/O Time',
    line2_label='POSIX I/O Bandwidth',
    time_col='io_time',
    x_num_ticks=8,
    y_num_ticks=5,
)

In [ ]:
fig, ax = analyzer.plots.xfer_size_timeline(
    figsize=(8, 3),
    unit='mb',
    x_num_ticks=8,
)